# 07 · Demo model trained on all 8 generators
Leave-one-generator-out showed that training on more generators generalises best (A2 detects a never-seen 8th generator with 95% accuracy). This notebook trains A2's head on **all 8** GenImage generators for the demo. The report keeps the official SD v1.4-only protocol (P1).

Run with **Save Version → Save & Run All (Commit)**. Download `fakelens_all8.zip` from the Output tab.

In [ ]:
# --- Kaggle setup (run first in every notebook) ---
# Settings: Accelerator = GPU (T4 x2 or P100), Internet = ON.
# Add Data: vtphatt2/GenImage-stable-diffusion-v1-4 and vtphatt2/GenImage-Validation
#           (+ the output dataset of the previous notebook, if you split stages across sessions).
import os, subprocess
REPO = "https://github.com/daniyal5293/fakelens.git"   # <- change, or upload the repo as a Kaggle dataset
if not os.path.exists("/kaggle/working/fakelens"):
    subprocess.run(["git", "clone", REPO, "/kaggle/working/fakelens"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/fakelens", "pull"], check=False)
os.chdir("/kaggle/working/fakelens")
!pip install -q open_clip_torch
!pip install -q -e . --no-deps
# Show the dataset folder structure (look for .../train/ai, .../train/nature, .../val/ai, .../val/nature):
!find /kaggle/input -maxdepth 7 -type d | head -80

In [ ]:
# Check the dataset folder names printed above and fix them here if they differ.
CFG = "configs/default.yaml"
SET = "--set roots.train_root=/kaggle/input/datasets/vtphatt2/genimage-stable-diffusion-v1-4 roots.test_root=/kaggle/input/datasets/vtphatt2/genimage-validation"
RUNS = "/kaggle/working/runs"

In [ ]:
# Use the committed manifest (identical splits to the main run); build it only if it is missing
import os
if not os.path.exists('data/manifests/manifest.csv'):
    !python -m fakelens.data.manifest --config $CFG $SET
!wc -l data/manifests/manifest.csv

In [ ]:
# CLIP ViT-L/14 features for the 8 generator pools and the test split (no SD v1.4 train split needed)
!python -m fakelens.features.extract --config $CFG $SET --backbone clip_vitl14 --splits logo test

In [ ]:
!python -m fakelens.train_head --config $CFG $SET --model A2 --experiment all8

In [ ]:
!mkdir -p checkpoints && cp $RUNS/all8/A2/seed0/head.pt checkpoints/A2_all8_head.pt
!cd /kaggle/working/fakelens && zip -qr /kaggle/working/fakelens_all8.zip results/all8 checkpoints/A2_all8_head.pt
!ls -lh /kaggle/working/fakelens_all8.zip

In [ ]:
import os
assert os.path.exists('/kaggle/working/fakelens_all8.zip'), 'Something failed - read the output above'
print('Done')